In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybedtools
from upsetplot import UpSet, from_memberships
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn2, venn3_circles
from collections import Counter

In [344]:
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
PREPROCESSING = Path("../preprocessing").resolve()
HIMORNA = DATA / "himorna"
BIOMART_INITIAL = DATA_INITIAL / "biomart"
GENCODE_INITIAL = DATA_INITIAL / "gencode"
GENCODE = DATA / "gencode"
ID_HISTORY_CONVERTER_RES = DATA / "id_history_converter_res"

In [20]:
def count_genome_coverage(anno, genome="hg38"):
    assembles_lenght = {"hg19":3101788170, "hg38":3099441038}
    anno_bed_sort = pybedtools.BedTool.from_dataframe(anno).sort()
    genome_coverage = anno_bed_sort.genome_coverage(bga=True, genome=genome)
    genome_coverage = genome_coverage.to_dataframe()
    genome_coverage["segment_lenght"] = genome_coverage["end"] - genome_coverage["start"]
    res = genome_coverage[genome_coverage["name"] != 0]["segment_lenght"].sum()
    return round(res / assembles_lenght[genome] * 100, 2)

In [406]:
def gencode_gtf_processing(to_gtf_file):
    gtf_file_GENCODE = to_gtf_file
    anno_GENCODE = pd.read_csv(gtf_file_GENCODE, sep="\t", header=None, skiprows=5)
    anno_GENCODE["ens_gene_primari"] = anno_GENCODE[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', ''))
    anno_GENCODE["ens_gene"] = anno_GENCODE[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])#Такой способ парсинга не учитывает гены на Y хромосоме в псевдоаутосомных регионах.
    anno_GENCODE["gene_name"] = anno_GENCODE[8].apply(lambda x: x.split("gene_name")[1].split(" ")[1].replace('"', '').replace(';', ''))
    anno_GENCODE["unique_genome_reg_name"] = anno_GENCODE["ens_gene"] + "-" + anno_GENCODE[2]
    gencode_hg38 = anno_GENCODE[[0, 3, 4, "unique_genome_reg_name", 5, 6, "ens_gene_primari", "ens_gene", "gene_name"]]
    mask = gencode_hg38["unique_genome_reg_name"].apply(lambda x: "gene" in x)
    gencode_genes_hg38 = gencode_hg38[mask]
    gencode_genes_hg38 = gencode_genes_hg38.rename(columns={0:"chrom", 3:"start", 4:"end", "unique_genome_reg_name":"name", 5:"score", 6:"strand"})
    gencode_genes_hg38 = gencode_genes_hg38.drop_duplicates()
    #gencode_genes_hg38["gene_ens--gene_name"] = gencode_genes_hg38["ens_gene"] + "--" + gencode_genes_hg38["gene_name"]
    return gencode_genes_hg38.reset_index(drop=True)

In [21]:
def create_bad_genes_history_df(to_txt):
    bad_genes_history = pd.read_csv(to_txt, skipinitialspace=True)
    bad_genes_history = bad_genes_history.rename(columns=lambda x: x.strip())
    mask = bad_genes_history["Old stable ID"].apply(lambda x: "ENSG" in x)
    bad_genes_history = bad_genes_history[mask].reset_index(drop=True)
    bad_genes_history["Old stable ID without v"] = bad_genes_history["Old stable ID"].apply(lambda x: x.split(".")[0])
    bad_genes_history["New stable ID without v"] = bad_genes_history["New stable ID"].apply(lambda x: x.split(".")[0])
    return bad_genes_history

In [22]:
def check_retired_conditions(df):
    # Находим все строки с <retired>
    df = df.reset_index(drop=True)
    retired_rows = df[df['New stable ID without v'] == '<retired>']
    if len(retired_rows) == 0:
        return False
    # Проверка 1: что <retired> только в одном экземпляре
    is_single_instance = len(retired_rows) == 1
    
    # Получаем индекс строки с <retired>
    retired_index = retired_rows.index[0]
    total_rows = len(df)
    
    # Проверка 2: что <retired> не на последней строке
    is_not_last = retired_index != total_rows - 1
    
    # Проверка 3: что <retired> не на предпоследней строке  
    is_not_second_last = retired_index != total_rows - 2
    if is_not_second_last == False:
        if df.iloc[retired_index, 2] == df.iloc[retired_index-1, 2]:
            is_not_second_last = True
    if is_single_instance & is_not_last & is_not_second_last:
        return True
    return False

In [23]:
#нужно модифицировать функцию для случаев, когда ген вырождается в несколько
def bad_history_analyzer(ensg_id, bad_genes_history):
    query_res = bad_genes_history.query(f"`Old stable ID without v` == '{ensg_id}'")
    query_res_length = len(query_res)
    if query_res_length == 0:
        new_ensg = "<retired>"
        return new_ensg
    if check_retired_conditions(df=query_res):
        new_ensg = "<retired>-to-many"
        return new_ensg
    else:
        new_ensg = query_res.iloc[-1, 5]
    return new_ensg

In [345]:
def jaccard_calculating(df1, df2):
    df1_sortbed = pybedtools.BedTool.from_dataframe(df1[["chrom","start","end","gene_ens--gene_name","score","strand"]]).sort()
    df2_sortbed = pybedtools.BedTool.from_dataframe(df2[["chrom","start","end","gene_ens--gene_name","score","strand"]]).sort()
    df1_overlap_df2 = df1_sortbed.intersect(df2_sortbed, wao=True, s=True)
    df1_overlap_df2 = df1_overlap_df2.to_dataframe(disable_auto_names=True, header=None)
    df1_overlap_df2 = df1_overlap_df2.rename(columns={0:"f-chrom", 1:"f-strart", 2:"f-end", 3:"f-name", 4:"f-score", 5:"f-strand",
                                                6:"g-chrom", 7:"g-strart", 8:"g-end", 9:"g-name", 10:"g-score", 11:"g-strand",
                                                12:"overlap"})
    df1_overlap_df2["merge_len"] = df1_overlap_df2[["f-strart", "f-end", "g-strart", "g-end"]].max(axis=1) - df1_overlap_df2[["f-strart", "f-end", "g-strart", "g-end"]].min(axis=1)
    df1_overlap_df2["jaccard"] = np.round(df1_overlap_df2["overlap"] / df1_overlap_df2["merge_len"], 2)
    df1_overlap_df2["g-name"] = df1_overlap_df2["g-name"].replace(".", "--")
    df1_overlap_df2["gene_en-metric"] = \
                df1_overlap_df2["f-name"].apply(lambda x: x.split("--")[0]) == df1_overlap_df2["g-name"].apply(lambda x: x.split("--")[0])
    df1_overlap_df2["gene_name-metric"] = \
                df1_overlap_df2["f-name"].apply(lambda x: x.split("--")[1]) == df1_overlap_df2["g-name"].apply(lambda x: x.split("--")[1])
    df1_overlap_df2["f-ens_name"] = df1_overlap_df2["f-name"].apply(lambda x: x.split("--")[0])
    df1_overlap_df2["g-ens_name"] = df1_overlap_df2["g-name"].apply(lambda x: x.split("--")[0])
    return df1_overlap_df2

p.s. В старых аннотация gencode гены в псевдоаутосомных областях на X и Y хромосомах имели один gene_name, а ensg идентификатор отличался лишь постфиксом для Y. В новых версиях каждому геномному локусу соотвествует свой уникальный идентификатор ensg. В идеале, конечно, стоило бы в данном ноутбуке получать идентификаторы для этих генов отдельно через индекс Жаккара. Однако, сейчас я этого делать не буду, поскольку в himorna порезаны идентификаторы генов (они split(".")[0])

In [407]:
to_file = GENCODE_INITIAL / "gencode.v31.basic.annotation.gtf.gz"
gencode_v31_genes_hg38 = gencode_gtf_processing(to_gtf_file=to_file)
gencode_v31_genes_hg38.shape

(60603, 9)

In [408]:
gencode_v31_genes_hg38.head(1)

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1


In [409]:
to_file = GENCODE_INITIAL / "gencode.v47.basic.annotation.gtf.gz"
gencode_v47_genes_hg38 = gencode_gtf_processing(to_gtf_file=to_file)
gencode_v47_genes_hg38.shape

(78724, 9)

In [410]:
gencode_v47_genes_hg38.head(1)

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825.2,ENSG00000290825,DDX11L16


In [411]:
mask = gencode_v31_genes_hg38["ens_gene_primari"].apply(lambda x: "PAR_Y" in x)
PAR_Y_indexes = gencode_v31_genes_hg38[mask].index
PAR_Y_indexes

Index([59999, 60000, 60001, 60002, 60003, 60004, 60005, 60006, 60007, 60008,
       60009, 60010, 60011, 60012, 60013, 60014, 60015, 60016, 60017, 60018,
       60019, 60020, 60021, 60022, 60023, 60024, 60025, 60026, 60027, 60028,
       60029, 60030, 60031, 60032, 60555, 60556, 60557, 60558, 60559, 60560,
       60561, 60562, 60563, 60564, 60565],
      dtype='int64')

In [412]:
len(PAR_Y_indexes)

45

In [413]:
for index in PAR_Y_indexes:
    gencode_v31_genes_hg38.loc[index, "ens_gene"] = gencode_v31_genes_hg38.loc[index, "ens_gene"] + "-PAR_Y"

In [414]:
mask = gencode_v31_genes_hg38["ens_gene_primari"].apply(lambda x: "PAR_Y" in x)
gencode_v31_genes_hg38[mask]

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name
59999,chrY,253743,255091,ENSG00000228572-gene,.,+,ENSG00000228572.7_PAR_Y,ENSG00000228572-PAR_Y,AL954722.1
60000,chrY,276322,303356,ENSG00000182378-gene,.,+,ENSG00000182378.14_PAR_Y,ENSG00000182378-PAR_Y,PLCXD1
60001,chrY,304529,318819,ENSG00000178605-gene,.,-,ENSG00000178605.13_PAR_Y,ENSG00000178605-PAR_Y,GTPBP6
60002,chrY,320990,321851,ENSG00000226179-gene,.,+,ENSG00000226179.6_PAR_Y,ENSG00000226179-PAR_Y,LINC00685
60003,chrY,333963,386955,ENSG00000167393-gene,.,-,ENSG00000167393.17_PAR_Y,ENSG00000167393-PAR_Y,PPP2R3B
60004,chrY,386980,405579,ENSG00000281849-gene,.,+,ENSG00000281849.3_PAR_Y,ENSG00000281849-PAR_Y,AL732314.6
60005,chrY,388100,388389,ENSG00000275287-gene,.,-,ENSG00000275287.5_PAR_Y,ENSG00000275287-PAR_Y,RF00017
60006,chrY,419157,421980,ENSG00000280767-gene,.,+,ENSG00000280767.3_PAR_Y,ENSG00000280767-PAR_Y,AL732314.4
60007,chrY,523775,524102,ENSG00000234958-gene,.,-,ENSG00000234958.6_PAR_Y,ENSG00000234958-PAR_Y,FABP5P13
60008,chrY,545236,545352,ENSG00000229232-gene,.,-,ENSG00000229232.6_PAR_Y,ENSG00000229232-PAR_Y,KRT18P53


In [416]:
gencode_v31_genes_hg38["gene_ens--gene_name"] = gencode_v31_genes_hg38["ens_gene"] + "--" + gencode_v31_genes_hg38["gene_name"]
gencode_v47_genes_hg38["gene_ens--gene_name"] = gencode_v47_genes_hg38["ens_gene"] + "--" + gencode_v47_genes_hg38["gene_name"]

Существует несколько подходов к решению проблемы соотнесения двух аннотаций. Универсального варианта нет, поэтому мы будем использовать комбинированный подход:

Если ensg между разметками не меняется - считаем, что пара найдена. \
Если в новой версии разметки такого гена нет, то запускаем id_history_converter. \
Дополнительно, если по версии id_history_converter идентификатор данного гена устарел ("retired"), перепроверяем его путем пересечения геномных координат двух разметок, вводя порог на индекс Жаккара.

p.s. Подход коллег в https://pmc.ncbi.nlm.nih.gov/articles/PMC12322893/ работает только при отображении близких во времени аннотаций. Есть множество примеров, когда восстановить ensg вовсе невозможно по трем используемым в статье метрикам даже с гораздо более слабым порогом на индекс Жаккара. Контакт для связи: Аглетдинов Матвей, @Mathwey

In [422]:
mask = gencode_v31_genes_hg38["ens_gene"].isin(gencode_v47_genes_hg38["ens_gene"])
bad_genes = gencode_v31_genes_hg38[~mask]["ens_gene"].unique()
len(bad_genes)

1369

Именно эти 1369 идентификатор генов необходимо перевести на gencode47, если это возможно. Для этого воспользуемся id_history_converter. \
Он входит в набор программ ensembl-tools: https://github.com/Ensembl/ensembl-tools/tree/release/99 \
Для уставки: \
git clone git@github.com:Ensembl/ensembl-tools.git \
mamba create -n ensembl-tools -y \
mamba install perl-app-cpanminus -y \
cpanm Bio::EnsEMBL

Также можно воспользоваться веб версией: \
https://www.ensembl.org/Homo_sapiens/Tools/IDMapper

In [423]:
with open(f"{ID_HISTORY_CONVERTER_RES}/bad-genes_id-gencode31_to_gencode47.txt", "w") as file:
    file.writelines("%s\n" % line for line in bad_genes)

In [424]:
to_txt = ID_HISTORY_CONVERTER_RES / "bad-genes_id-gencode31_to_gencode47-history.txt"
bad_genes_history = create_bad_genes_history_df(to_txt)
bad_genes_history

,Old stable ID,New stable ID,Release,Mapping score,Old stable ID without v,New stable ID without v
0,ENSG00000238009.1,ENSG00000238009.2,56,0.25645,ENSG00000238009,ENSG00000238009
1,ENSG00000238009.2,ENSG00000238009.3,76,0.973847,ENSG00000238009,ENSG00000238009
2,ENSG00000238009.3,ENSG00000238009.4,77,1,ENSG00000238009,ENSG00000238009
3,ENSG00000238009.4,ENSG00000238009.5,79,1,ENSG00000238009,ENSG00000238009
4,ENSG00000238009.5,ENSG00000238009.6,81,1,ENSG00000238009,ENSG00000238009
...,...,...,...,...,...,...
6292,ENSG00000226362.2,ENSG00000226362.2,98,0.9,ENSG00000226362,ENSG00000226362
6293,ENSG00000226362.2,ENSG00000226362.2,99,0.9,ENSG00000226362,ENSG00000226362
6294,ENSG00000226362.2,ENSG00000226362.2,100,0.9,ENSG00000226362,ENSG00000226362
6295,ENSG00000226362.2,ENSG00000290726.2,111,0.99,ENSG00000226362,ENSG00000290726


In [425]:
bad_genes_history["Old stable ID without v"].nunique()

1320

In [426]:
len(bad_genes)

1369

По какой-то причне эти 4 идентификатора (помимо *-PAR_Y) не вошли в таблицу результатов. Полагаем, что они также устарели.

In [427]:
set(bad_genes) - set(bad_genes_history["Old stable ID without v"].unique())

{'ENSG00000002586-PAR_Y',
 'ENSG00000124333-PAR_Y',
 'ENSG00000124334-PAR_Y',
 'ENSG00000167393-PAR_Y',
 'ENSG00000168939-PAR_Y',
 'ENSG00000169084-PAR_Y',
 'ENSG00000169093-PAR_Y',
 'ENSG00000169100-PAR_Y',
 'ENSG00000178605-PAR_Y',
 'ENSG00000182162-PAR_Y',
 'ENSG00000182378-PAR_Y',
 'ENSG00000182484-PAR_Y',
 'ENSG00000185203-PAR_Y',
 'ENSG00000185291-PAR_Y',
 'ENSG00000185960-PAR_Y',
 'ENSG00000196433-PAR_Y',
 'ENSG00000197976-PAR_Y',
 'ENSG00000198223-PAR_Y',
 'ENSG00000205755-PAR_Y',
 'ENSG00000214717-PAR_Y',
 'ENSG00000223274-PAR_Y',
 'ENSG00000223484-PAR_Y',
 'ENSG00000223511-PAR_Y',
 'ENSG00000223571-PAR_Y',
 'ENSG00000223773-PAR_Y',
 'ENSG00000225661-PAR_Y',
 'ENSG00000226179-PAR_Y',
 'ENSG00000227159-PAR_Y',
 'ENSG00000228410-PAR_Y',
 'ENSG00000228572-PAR_Y',
 'ENSG00000229232-PAR_Y',
 'ENSG00000230542-PAR_Y',
 'ENSG00000234622-PAR_Y',
 'ENSG00000234958-PAR_Y',
 'ENSG00000236017-PAR_Y',
 'ENSG00000236871-PAR_Y',
 'ENSG00000237040-PAR_Y',
 'ENSG00000237531-PAR_Y',
 'ENSG000002

In [428]:
bad_genes_resolutions = {}
old_ids = bad_genes_history["Old stable ID without v"].unique()
for ensg_id in bad_genes:
    if ensg_id in old_ids:
        bad_genes_resolutions[ensg_id] = bad_history_analyzer(
            ensg_id=ensg_id, 
            bad_genes_history=bad_genes_history
        )
    else:
        bad_genes_resolutions[ensg_id] = "<retired>"

In [429]:
len(set(bad_genes))

1369

In [430]:
df_bad_genes_resolutions = pd.DataFrame({"name":bad_genes_resolutions.keys(), "name_latest":bad_genes_resolutions.values()})
df_bad_genes_resolutions

,name,name_latest
0,ENSG00000238009,ENSG00000241860
1,ENSG00000230699,ENSG00000272438
2,ENSG00000241180,ENSG00000272438
3,ENSG00000236948,ENSG00000289893
4,ENSG00000277726,<retired>
...,...,...
1364,ENSG00000124334-PAR_Y,<retired>
1365,ENSG00000270726-PAR_Y,<retired>
1366,ENSG00000185203-PAR_Y,<retired>
1367,ENSG00000182484-PAR_Y,<retired>


In [431]:
df_bad_genes_resolutions["name_latest2"] = np.where(df_bad_genes_resolutions["name"] == df_bad_genes_resolutions["name_latest"], "<retired>", df_bad_genes_resolutions["name_latest"])

In [432]:
df_bad_genes_resolutions

,name,name_latest,name_latest2
0,ENSG00000238009,ENSG00000241860,ENSG00000241860
1,ENSG00000230699,ENSG00000272438,ENSG00000272438
2,ENSG00000241180,ENSG00000272438,ENSG00000272438
3,ENSG00000236948,ENSG00000289893,ENSG00000289893
4,ENSG00000277726,<retired>,<retired>
...,...,...,...
1364,ENSG00000124334-PAR_Y,<retired>,<retired>
1365,ENSG00000270726-PAR_Y,<retired>,<retired>
1366,ENSG00000185203-PAR_Y,<retired>,<retired>
1367,ENSG00000182484-PAR_Y,<retired>,<retired>


In [433]:
mask = df_bad_genes_resolutions["name_latest2"].apply(lambda x: "ENSG" in x)
name_latest2_unique_df = pd.DataFrame(df_bad_genes_resolutions[mask]["name_latest2"].unique())
len(name_latest2_unique_df)

805

Проверим, все ли новые идентификаторы есть в gencode47

In [434]:
mask = name_latest2_unique_df[0].isin(gencode_v47_genes_hg38["ens_gene"])
name_latest2_unique_df[~mask]

,0
119,ENSG00000233143
398,ENSG00000225489


Восстановим актуальный идентификатор для этих генов руками:
ENSG00000233143 - ENSG00000223874
ENSG00000225489 - ENSG00000236924

In [435]:
assert "ENSG00000223874" in gencode_v47_genes_hg38["ens_gene"].to_list()
assert "ENSG00000236924" in gencode_v47_genes_hg38["ens_gene"].to_list()

In [436]:
df_bad_genes_resolutions[df_bad_genes_resolutions["name_latest2"] == "ENSG00000233143"]

,name,name_latest,name_latest2
181,ENSG00000236886,ENSG00000233143,ENSG00000233143
183,ENSG00000286601,ENSG00000233143,ENSG00000233143


In [437]:
df_bad_genes_resolutions[df_bad_genes_resolutions["name_latest2"] == "ENSG00000225489"]

,name,name_latest,name_latest2
635,ENSG00000229611,ENSG00000225489,ENSG00000225489


In [438]:
df_bad_genes_resolutions.iloc[181, 2] = "ENSG00000223874"
df_bad_genes_resolutions.iloc[183, 2] = "ENSG00000223874"
df_bad_genes_resolutions.iloc[635, 2] = "ENSG00000236924"

In [439]:
#Проверим, нет ли неотловленных состояний one-to-many (<retired>-to-many)
mask = df_bad_genes_resolutions["name"].duplicated()
assert mask.sum() == 0

In [440]:
mask = df_bad_genes_resolutions["name"] == df_bad_genes_resolutions["name_latest2"]
mask.sum()

0

In [441]:
df_bad_genes_resolutions

,name,name_latest,name_latest2
0,ENSG00000238009,ENSG00000241860,ENSG00000241860
1,ENSG00000230699,ENSG00000272438,ENSG00000272438
2,ENSG00000241180,ENSG00000272438,ENSG00000272438
3,ENSG00000236948,ENSG00000289893,ENSG00000289893
4,ENSG00000277726,<retired>,<retired>
...,...,...,...
1364,ENSG00000124334-PAR_Y,<retired>,<retired>
1365,ENSG00000270726-PAR_Y,<retired>,<retired>
1366,ENSG00000185203-PAR_Y,<retired>,<retired>
1367,ENSG00000182484-PAR_Y,<retired>,<retired>


In [442]:
mask = df_bad_genes_resolutions["name_latest2"].apply(lambda x: "retired" in x)
bad_genes_retired_gencode31 = df_bad_genes_resolutions[mask]["name"]
len(bad_genes_retired_gencode31)

415

Подсчет индекса Жаккара:

In [443]:
gencode_v31_overlap_gencode_v47 = jaccard_calculating(
    df1=gencode_v31_genes_hg38[["chrom","start","end","gene_ens--gene_name","score","strand"]],
    df2=gencode_v47_genes_hg38[["chrom","start","end","gene_ens--gene_name","score","strand"]]
)
gencode_v31_overlap_gencode_v47

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,g-score,g-strand,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name
0,chr1,11869,14409,ENSG00000223972--DDX11L1,.,+,chr1,12010,13670,ENSG00000223972--DDX11L1,.,+,1660,2540,0.65,True,True,ENSG00000223972,ENSG00000223972
1,chr1,11869,14409,ENSG00000223972--DDX11L1,.,+,chr1,11121,24894,ENSG00000290825--DDX11L16,.,+,2540,13773,0.18,False,False,ENSG00000223972,ENSG00000290825
2,chr1,14404,29570,ENSG00000227232--WASH7P,.,-,chr1,17369,17436,ENSG00000278267--MIR6859-1,.,-,67,15166,0.00,False,False,ENSG00000227232,ENSG00000278267
3,chr1,14404,29570,ENSG00000227232--WASH7P,.,-,chr1,14356,30744,ENSG00000310526--WASH7P,.,-,15166,16388,0.93,False,True,ENSG00000227232,ENSG00000310526
4,chr1,14404,29570,ENSG00000227232--WASH7P,.,-,chr1,14696,24886,ENSG00000227232--WASH7P,.,-,10190,15166,0.67,True,True,ENSG00000227232,ENSG00000227232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91878,chrY,57201143,57203357,ENSG00000185203-PAR_Y--WASIR1,.,-,chrY,57201135,57203405,ENSG00000292370--WASIR1,.,-,2214,2270,0.98,False,True,ENSG00000185203-PAR_Y,ENSG00000292370
91879,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57207346,57212230,ENSG00000292372--WASH6P,.,+,4884,4884,1.00,False,True,ENSG00000182484-PAR_Y,ENSG00000292372
91880,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57190738,57208756,ENSG00000292369--ENSG00000292369,.,+,1410,21492,0.07,False,False,ENSG00000182484-PAR_Y,ENSG00000292369
91881,chrY,57212184,57214397,ENSG00000227159-PAR_Y--DDX11L16,.,-,chrY,57208516,57215634,ENSG00000310542--ENSG00000310542,.,-,2213,7118,0.31,False,False,ENSG00000227159-PAR_Y,ENSG00000310542


In [444]:
mask = gencode_v31_overlap_gencode_v47["f-ens_name"].apply(lambda x: "ENSG00000277120" in x)
gencode_v31_overlap_gencode_v47[mask]

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,g-score,g-strand,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name
87745,chrX,2609191,2609254,ENSG00000277120--MIR6089,.,+,chrX,2609191,2609254,ENSG00000277120--MIR6089,.,+,63,63,1.0,True,True,ENSG00000277120,ENSG00000277120
87746,chrX,2609191,2609254,ENSG00000277120--MIR6089,.,+,chrX,2608406,2666596,ENSG00000291040--CD99P1,.,+,63,58190,0.0,False,False,ENSG00000277120,ENSG00000291040
91142,chrY,2609191,2609254,ENSG00000277120-PAR_Y--MIR6089,.,+,chrY,2609191,2609254,ENSG00000292346--MIR6089,.,+,63,63,1.0,False,True,ENSG00000277120-PAR_Y,ENSG00000292346
91143,chrY,2609191,2609254,ENSG00000277120-PAR_Y--MIR6089,.,+,chrY,2608406,2666596,ENSG00000292362--CD99P1,.,+,63,58190,0.0,False,False,ENSG00000277120-PAR_Y,ENSG00000292362


Полностью полагаться на соответствие gene_name между сборками нельзя. По ним есть множество дублей:

In [445]:
gencode_v47_genes_hg38.groupby(by="gene_name").aggregate(
    Число_дублей=("ens_gene","nunique")
).query("Число_дублей >= 2").sort_values(by="Число_дублей", ascending=False)

,Число_дублей
gene_name,
Y_RNA,756
Metazoa_SRP,170
U3,50
U6,33
SNORA70,27
...,...
FRG1HP,2
FRG1BP,2
FOLH1B,2


Будем работать только с теми генами, которые по версии id_history_converter имеют статус "retired" \
Для них попробуем восстановить gene_ens, вводя порог на индекс Жакара >= 0.99

In [446]:
mask = df_bad_genes_resolutions["name_latest2"].apply(lambda x: x == "<retired>")
df_bad_genes_resolutions_retired = df_bad_genes_resolutions[mask]
df_bad_genes_resolutions_retired

,name,name_latest,name_latest2
4,ENSG00000277726,<retired>,<retired>
5,ENSG00000271895,<retired>,<retired>
9,ENSG00000234166,<retired>,<retired>
12,ENSG00000225986,<retired>,<retired>
17,ENSG00000116883,<retired>,<retired>
...,...,...,...
1364,ENSG00000124334-PAR_Y,<retired>,<retired>
1365,ENSG00000270726-PAR_Y,<retired>,<retired>
1366,ENSG00000185203-PAR_Y,<retired>,<retired>
1367,ENSG00000182484-PAR_Y,<retired>,<retired>


In [448]:
gencode_v31_overlap_gencode_v47_bad_genes_retired = pd.merge(left=gencode_v31_overlap_gencode_v47, right=df_bad_genes_resolutions_retired, how="inner", left_on="f-ens_name", right_on="name")
gencode_v31_overlap_gencode_v47_bad_genes_retired

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,...,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name,name,name_latest,name_latest2
0,chr1,11012662,11030528,ENSG00000277726--AL109811.3,.,+,chr1,11012344,11030528,ENSG00000120948--TARDBP,...,17866,18184,0.98,False,False,ENSG00000277726,ENSG00000120948,ENSG00000277726,<retired>,<retired>
1,chr1,11029659,11030528,ENSG00000271895--AL109811.2,.,+,chr1,11012344,11030528,ENSG00000120948--TARDBP,...,869,18184,0.05,False,False,ENSG00000271895,ENSG00000120948,ENSG00000271895,<retired>,<retired>
2,chr1,16197854,16198357,ENSG00000234166--ARHGEF19-AS1,.,+,.,-1,-1,--,...,0,16198358,0.00,False,False,ENSG00000234166,,ENSG00000234166,<retired>,<retired>
3,chr1,20184242,20186486,ENSG00000225986--UBXN10-AS1,.,-,chr1,20161253,20186518,ENSG00000187980--PLA2G2C,...,2244,25265,0.09,False,False,ENSG00000225986,ENSG00000187980,ENSG00000225986,<retired>,<retired>
4,chr1,36323734,36329221,ENSG00000116883--AL591845.1,.,+,chr1,36306368,36329340,ENSG00000214193--SH3D21,...,5487,22972,0.24,False,False,ENSG00000116883,ENSG00000214193,ENSG00000116883,<retired>,<retired>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526,chrY,57201143,57203357,ENSG00000185203-PAR_Y--WASIR1,.,-,chrY,57201135,57203405,ENSG00000292370--WASIR1,...,2214,2270,0.98,False,True,ENSG00000185203-PAR_Y,ENSG00000292370,ENSG00000185203-PAR_Y,<retired>,<retired>
527,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57207346,57212230,ENSG00000292372--WASH6P,...,4884,4884,1.00,False,True,ENSG00000182484-PAR_Y,ENSG00000292372,ENSG00000182484-PAR_Y,<retired>,<retired>
528,chrY,57207346,57212230,ENSG00000182484-PAR_Y--WASH6P,.,+,chrY,57190738,57208756,ENSG00000292369--ENSG00000292369,...,1410,21492,0.07,False,False,ENSG00000182484-PAR_Y,ENSG00000292369,ENSG00000182484-PAR_Y,<retired>,<retired>
529,chrY,57212184,57214397,ENSG00000227159-PAR_Y--DDX11L16,.,-,chrY,57208516,57215634,ENSG00000310542--ENSG00000310542,...,2213,7118,0.31,False,False,ENSG00000227159-PAR_Y,ENSG00000310542,ENSG00000227159-PAR_Y,<retired>,<retired>


In [494]:
gencode_v31_overlap_gencode_v47_bad_genes_retired = gencode_v31_overlap_gencode_v47_bad_genes_retired.drop_duplicates()
len(gencode_v31_overlap_gencode_v47_bad_genes_retired)

531

In [496]:
mask = (gencode_v31_overlap_gencode_v47_bad_genes_retired["jaccard"] >= 0.99)
gencode_v31_overlap_gencode_v47_bad_genes_retired_jaccard_ok = gencode_v31_overlap_gencode_v47_bad_genes_retired[mask]
gencode_v31_overlap_gencode_v47_bad_genes_retired_jaccard_ok.head(3)

,f-chrom,f-strart,f-end,f-name,f-score,f-strand,g-chrom,g-strart,g-end,g-name,...,overlap,merge_len,jaccard,gene_en-metric,gene_name-metric,f-ens_name,g-ens_name,name,name_latest,name_latest2
30,chr1,168695874,169103276,ENSG00000285622--AL135926.1,.,-,chr1,168695874,169103307,ENSG00000203601--LINC00970,...,407402,407433,1.0,False,False,ENSG00000285622,ENSG00000203601,ENSG00000285622,ENSG00000285622,<retired>
33,chr1,182944365,182944490,ENSG00000199837--RNA5SP71,.,-,chr1,182944365,182944490,ENSG00000285609--5S_rRNA,...,125,125,1.0,False,False,ENSG00000199837,ENSG00000285609,ENSG00000199837,<retired>,<retired>
72,chr11,797511,799185,ENSG00000274897--PANO1,.,+,chr11,797511,799190,ENSG00000288675--PANO1,...,1674,1679,1.0,False,True,ENSG00000274897,ENSG00000288675,ENSG00000274897,<retired>,<retired>


In [497]:
len(gencode_v31_overlap_gencode_v47_bad_genes_retired_jaccard_ok)

60

In [498]:
#Проверяем на множественные пересечения. Если падает с ошибкой, то нужно выбрать лучший вариант пересечения (возможно, тогда стоит дополнительно посмотреть на gene_name)
assert gencode_v31_overlap_gencode_v47_bad_genes_retired_jaccard_ok["f-name"].duplicated().sum() == 0

In [501]:
df_bad_genes_resolutions_jaccar_corrected = pd.merge(left=df_bad_genes_resolutions, right=gencode_v31_overlap_gencode_v47_bad_genes_retired_jaccard_ok[["f-ens_name","g-ens_name"]], how="left", left_on="name", right_on="f-ens_name")
df_bad_genes_resolutions_jaccar_corrected

,name,name_latest,name_latest2,f-ens_name,g-ens_name
0,ENSG00000238009,ENSG00000241860,ENSG00000241860,NaN,NaN
1,ENSG00000230699,ENSG00000272438,ENSG00000272438,NaN,NaN
2,ENSG00000241180,ENSG00000272438,ENSG00000272438,NaN,NaN
3,ENSG00000236948,ENSG00000289893,ENSG00000289893,NaN,NaN
4,ENSG00000277726,<retired>,<retired>,NaN,NaN
...,...,...,...,...,...
1364,ENSG00000124334-PAR_Y,<retired>,<retired>,NaN,NaN
1365,ENSG00000270726-PAR_Y,<retired>,<retired>,ENSG00000270726-PAR_Y,ENSG00000292369
1366,ENSG00000185203-PAR_Y,<retired>,<retired>,NaN,NaN
1367,ENSG00000182484-PAR_Y,<retired>,<retired>,ENSG00000182484-PAR_Y,ENSG00000292372


In [502]:
df_bad_genes_resolutions_jaccar_corrected["g-ens_name"] = df_bad_genes_resolutions_jaccar_corrected["g-ens_name"].fillna(df_bad_genes_resolutions_jaccar_corrected["name_latest2"])

In [503]:
mask = df_bad_genes_resolutions_jaccar_corrected["name_latest2"] == df_bad_genes_resolutions_jaccar_corrected["g-ens_name"]
len(df_bad_genes_resolutions_jaccar_corrected[~mask])

60

In [504]:
df_bad_genes_resolutions_jaccar_corrected = df_bad_genes_resolutions_jaccar_corrected.rename(columns={"g-ens_name":"name_latest3"})
df_bad_genes_resolutions_jaccar_corrected = df_bad_genes_resolutions_jaccar_corrected[["name", "name_latest2", "name_latest3"]]
df_bad_genes_resolutions_jaccar_corrected

,name,name_latest2,name_latest3
0,ENSG00000238009,ENSG00000241860,ENSG00000241860
1,ENSG00000230699,ENSG00000272438,ENSG00000272438
2,ENSG00000241180,ENSG00000272438,ENSG00000272438
3,ENSG00000236948,ENSG00000289893,ENSG00000289893
4,ENSG00000277726,<retired>,<retired>
...,...,...,...
1364,ENSG00000124334-PAR_Y,<retired>,<retired>
1365,ENSG00000270726-PAR_Y,<retired>,ENSG00000292369
1366,ENSG00000185203-PAR_Y,<retired>,<retired>
1367,ENSG00000182484-PAR_Y,<retired>,ENSG00000292372


In [505]:
mask = df_bad_genes_resolutions_jaccar_corrected["name_latest2"].apply(lambda x: "ENSG" in x)
len(df_bad_genes_resolutions_jaccar_corrected[mask]["name"].unique())

954

In [506]:
mask = df_bad_genes_resolutions_jaccar_corrected["name_latest3"].apply(lambda x: "ENSG" in x)
len(df_bad_genes_resolutions_jaccar_corrected[mask]["name"].unique())

1014

Таким образом, в итоге нам удалось восстановить 1014 ens_gene.

In [507]:
premerged_1 = gencode_v31_genes_hg38.copy()
premerged_1["ens_gene-unchangeable"] = np.where(
    gencode_v31_genes_hg38["ens_gene"].isin(gencode_v47_genes_hg38["ens_gene"]),
    gencode_v31_genes_hg38["ens_gene"],
    np.nan
)
premerged_1

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name,gene_ens--gene_name,ens_gene-unchangeable
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1,ENSG00000223972
1,chr1,14404,29570,ENSG00000227232-gene,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P,ENSG00000227232
2,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1,ENSG00000278267
3,chr1,29554,31109,ENSG00000243485-gene,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485--MIR1302-2HG,ENSG00000243485
4,chr1,30366,30503,ENSG00000284332-gene,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332--MIR1302-2,ENSG00000284332
...,...,...,...,...,...,...,...,...,...,...,...
60598,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6,ENSG00000198695
60599,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE,ENSG00000210194
60600,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB,ENSG00000198727
60601,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT,ENSG00000210195


In [508]:
premerged_1["ens_gene-unchangeable"].isna().sum()

1369

gencode_v47_genes_hg38_temp = gencode_v47_genes_hg38.copy()
gencode_v47_genes_hg38_temp["ens_gene-temp"] = gencode_v47_genes_hg38_temp["ens_gene"]
premerged_1 = pd.merge(left=gencode_v31_genes_hg38, right=gencode_v47_genes_hg38_temp[["ens_gene", "ens_gene-temp"]], how="left", left_on="ens_gene", right_on="ens_gene")
premerged_1

premerged_1["ens_gene-temp"].isna().sum()

premerged_1 = pd.merge(left=gencode_v31_genes_hg38, right=gencode_v47_genes_hg38[["ens_gene"]], how="left", left_on="ens_gene", right_on="ens_gene", indicator=True)
premerged_1["ens_gene-unchangeable"] = np.where(premerged_1["_merge"] == "both", premerged_1["ens_gene"], np.nan)
premerged_1

mask = gencode_v31_genes_hg38["ens_gene"].isin(gencode_v47_genes_hg38["ens_gene"])
premerged_1 = gencode_v31_genes_hg38[mask][["chrom","start","end","score","ens_gene","gene_name","strand"]]
premerged_1

In [509]:
premerged_2 = pd.merge(left=gencode_v31_genes_hg38, right=df_bad_genes_resolutions_jaccar_corrected[["name","name_latest3"]], how="left", left_on="ens_gene", right_on="name")
premerged_2

,chrom,start,end,name_x,score,strand,ens_gene_primari,ens_gene,gene_name,gene_ens--gene_name,name_y,name_latest3
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1,NaN,NaN
1,chr1,14404,29570,ENSG00000227232-gene,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232--WASH7P,NaN,NaN
2,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267--MIR6859-1,NaN,NaN
3,chr1,29554,31109,ENSG00000243485-gene,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485--MIR1302-2HG,NaN,NaN
4,chr1,30366,30503,ENSG00000284332-gene,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332--MIR1302-2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
60598,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695--MT-ND6,NaN,NaN
60599,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194--MT-TE,NaN,NaN
60600,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727--MT-CYB,NaN,NaN
60601,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195--MT-TT,NaN,NaN


In [510]:
premerged_2["name_latest3"].isna().sum()

59234

In [515]:
59234 + 1369

60603

In [468]:
gencode_v31_to_v47_genes_hg38 = gencode_v31_genes_hg38.copy()
gencode_v31_to_v47_genes_hg38.head(1)

,chrom,start,end,name,score,strand,ens_gene_primari,ens_gene,gene_name,gene_ens--gene_name
0,chr1,11869,14409,ENSG00000223972-gene,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972--DDX11L1


In [469]:
gencode_v31_to_v47_genes_hg38["new-ens_gene"] = premerged_1["ens_gene-unchangeable"].fillna(premerged_2["name_latest3"])

In [470]:
assert gencode_v31_to_v47_genes_hg38["new-ens_gene"].isna().sum() == 0

In [471]:
del gencode_v31_to_v47_genes_hg38["name"]
del gencode_v31_to_v47_genes_hg38["gene_ens--gene_name"]

In [472]:
gencode_v31_to_v47_genes_hg38

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267
3,chr1,29554,31109,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485
4,chr1,30366,30503,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332
...,...,...,...,...,...,...,...,...,...
60598,chrM,14149,14673,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695
60599,chrM,14674,14742,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194
60600,chrM,14747,15887,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727
60601,chrM,15888,15953,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195


In [473]:
mask = gencode_v31_to_v47_genes_hg38["ens_gene"] != gencode_v31_to_v47_genes_hg38["new-ens_gene"]
gencode_v31_to_v47_genes_hg38[mask]["ens_gene"].nunique()

1369

In [474]:
mask = (gencode_v31_to_v47_genes_hg38["ens_gene"] != gencode_v31_to_v47_genes_hg38["new-ens_gene"]) & (gencode_v31_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: "ENSG" in x))
gencode_v31_to_v47_genes_hg38[mask]["ens_gene"].nunique()

1014

In [475]:
mask = gencode_v31_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: "ENSG" not in x)
len(gencode_v31_to_v47_genes_hg38[mask])

355

In [476]:
mask = (gencode_v31_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: x == "<retired>"))
gencode_v31_to_v47_genes_hg38[mask]["ens_gene"].nunique()

349

In [477]:
mask = (gencode_v31_to_v47_genes_hg38["new-ens_gene"].apply(lambda x: x == "<retired>-to-many"))
gencode_v31_to_v47_genes_hg38[mask]["ens_gene"].nunique()

6

Таким образом: \
1369 генов претерпели изменение \
Для 1014 из них удалось восстановить актуальный идентификатор в gencode47 \
355 устарели, найти соответствие для них не получилось. Причем 349 просто устарели, а 6 отобразились в несколько актуальных идентификаторов ("one-to-many")

Теперь добавим идентификаторы gene_name:

In [478]:
gencode_v31_to_v47_genes_hg38

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267
3,chr1,29554,31109,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485
4,chr1,30366,30503,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332
...,...,...,...,...,...,...,...,...,...
60598,chrM,14149,14673,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695
60599,chrM,14674,14742,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194
60600,chrM,14747,15887,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727
60601,chrM,15888,15953,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195


In [479]:
gencode_v31_to_v47_genes_hg38_with_names = pd.merge(left=gencode_v31_to_v47_genes_hg38, right=gencode_v47_genes_hg38[["ens_gene","gene_name"]], how="left", left_on="new-ens_gene", right_on="ens_gene")
del gencode_v31_to_v47_genes_hg38_with_names["ens_gene_y"]
gencode_v31_to_v47_genes_hg38_with_names = gencode_v31_to_v47_genes_hg38_with_names.rename(columns={"ens_gene_x":"ens_gene","gene_name_x":"gene_name", "gene_name_y":"new-gene_name"})

In [480]:
mask = gencode_v31_to_v47_genes_hg38_with_names["new-gene_name"].isna()
gencode_v31_to_v47_genes_hg38_with_names[mask]

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
373,chr1,11012662,11030528,.,+,ENSG00000277726.4,ENSG00000277726,AL109811.3,<retired>,NaN
375,chr1,11029659,11030528,.,+,ENSG00000271895.2,ENSG00000271895,AL109811.2,<retired>,NaN
547,chr1,16197854,16198357,.,+,ENSG00000234166.1,ENSG00000234166,ARHGEF19-AS1,<retired>,NaN
657,chr1,20184242,20186486,.,-,ENSG00000225986.1,ENSG00000225986,UBXN10-AS1,<retired>,NaN
1179,chr1,36323734,36329221,.,+,ENSG00000116883.8,ENSG00000116883,AL591845.1,<retired>,NaN
...,...,...,...,...,...,...,...,...,...,...
60030,chrY,2609348,2657229,.,+,ENSG00000223773.7_PAR_Y,ENSG00000223773-PAR_Y,CD99P1,<retired>,NaN
60329,chrY,19076946,19077416,.,-,ENSG00000286265.1,ENSG00000286265,AC007244.1,<retired>,NaN
60555,chrY,56954332,56968979,.,+,ENSG00000168939.11_PAR_Y,ENSG00000168939-PAR_Y,SPRY3,<retired>,NaN
60561,chrY,57184101,57197337,.,+,ENSG00000124334.17_PAR_Y,ENSG00000124334-PAR_Y,IL9R,<retired>,NaN


In [482]:
gencode_v31_to_v47_genes_hg38_with_names["gene_name"].isna().sum()

0

Заполним пустые ячейки в new-gene_name продублированными new-ens_gene. Точно также сделано в gencode47. Даже если для ensg гена нет gene_name, для него просто продублировано ens_gene.

In [483]:
gencode_v31_to_v47_genes_hg38_with_names["new-gene_name"] = gencode_v31_to_v47_genes_hg38_with_names["new-gene_name"].fillna(gencode_v31_to_v47_genes_hg38_with_names["new-ens_gene"])

In [484]:
gencode_v31_to_v47_genes_hg38_with_names

,chrom,start,end,score,strand,ens_gene_primari,ens_gene,gene_name,new-ens_gene,new-gene_name
0,chr1,11869,14409,.,+,ENSG00000223972.5,ENSG00000223972,DDX11L1,ENSG00000223972,DDX11L1
1,chr1,14404,29570,.,-,ENSG00000227232.5,ENSG00000227232,WASH7P,ENSG00000227232,WASH7P
2,chr1,17369,17436,.,-,ENSG00000278267.1,ENSG00000278267,MIR6859-1,ENSG00000278267,MIR6859-1
3,chr1,29554,31109,.,+,ENSG00000243485.5,ENSG00000243485,MIR1302-2HG,ENSG00000243485,MIR1302-2HG
4,chr1,30366,30503,.,+,ENSG00000284332.1,ENSG00000284332,MIR1302-2,ENSG00000284332,MIR1302-2
...,...,...,...,...,...,...,...,...,...,...
60598,chrM,14149,14673,.,-,ENSG00000198695.2,ENSG00000198695,MT-ND6,ENSG00000198695,MT-ND6
60599,chrM,14674,14742,.,-,ENSG00000210194.1,ENSG00000210194,MT-TE,ENSG00000210194,MT-TE
60600,chrM,14747,15887,.,+,ENSG00000198727.2,ENSG00000198727,MT-CYB,ENSG00000198727,MT-CYB
60601,chrM,15888,15953,.,+,ENSG00000210195.2,ENSG00000210195,MT-TT,ENSG00000210195,MT-TT


In [485]:
gencode_v31_to_v47_genes_hg38_with_names["new-ens_gene"].isna().sum()

0

In [486]:
gencode_v31_to_v47_genes_hg38_with_names["new-gene_name"].isna().sum()

0

In [487]:
to_save = GENCODE / 'gencode_v31-to-v47-genes-hg38-with_names-all_info.tsv'
gencode_v31_to_v47_genes_hg38_with_names.to_csv(to_save, sep="\t", index=False)